# 03 - Carga de datos en Snowflake
## Contratos Menores - Ayuntamiento de Vilagarcía de Arousa

Este notebook carga el CSV limpio (producto del notebook 02) en Snowflake usando un **esquema en estrella**.

**Entrada:** `data/processed/contratos_limpios.csv`  
**Destino:** 2 tablas de dimensiones + `FACT_CONTRATOS` en Snowflake

Las credenciales se leen del fichero `.env` — nunca se escriben directamente en el código.

**Convenciones de diseño:**
- La FK temporal en `FACT_CONTRATOS` se llama `fecha` (igual que la PK de `DIM_FECHA`) para que Power BI detecte la relación automáticamente.
- `DIM_FECHA` contiene un calendario **completo y continuo** (todos los días entre la fecha mínima y máxima), requisito para poder marcarla como tabla de fechas en Power BI.

## 0 — Instalar dependencias
Ejecuta solo la primera vez.

In [15]:
# Instalamos las dependencias necesarias:
# - snowflake-connector-python: conector oficial para comunicarse con Snowflake
# - python-dotenv: para leer las credenciales desde el fichero .env
# - pandas: para manipular los datos antes de cargarlos
# --prefer-binary descarga ruedas precompiladas en lugar de compilar desde fuente (más rápido)
# -q suprime la salida verbosa de pip
%pip install --prefer-binary -q snowflake-connector-python python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.


## 1 — Importaciones y carga de credenciales

In [16]:
import os                        # para leer variables de entorno
import pandas as pd              # manipulación de datos en memoria
import snowflake.connector       # conector oficial de Snowflake para Python
from pathlib import Path         # rutas de ficheros multiplataforma
from dotenv import load_dotenv   # carga variables desde el fichero .env

# Cargamos el fichero .env que está un nivel arriba del notebook
# Esto inyecta las variables (SNOWFLAKE_ACCOUNT, etc.) en el entorno del proceso
load_dotenv(Path("../.env"))

# Leemos cada credencial del entorno con os.getenv()
# Si la variable no existe en .env, devuelve None en lugar de lanzar un error
ACCOUNT   = os.getenv("SNOWFLAKE_ACCOUNT")   # identificador de la cuenta Snowflake
USER      = os.getenv("SNOWFLAKE_USER")      # nombre de usuario
PASSWORD  = os.getenv("SNOWFLAKE_PASSWORD")  # contraseña
WAREHOUSE = os.getenv("SNOWFLAKE_WAREHOUSE") # clúster de cómputo que ejecutará las queries
DATABASE  = os.getenv("SNOWFLAKE_DATABASE")  # base de datos destino
SCHEMA    = os.getenv("SNOWFLAKE_SCHEMA")    # esquema dentro de la base de datos

# Construimos una lista con los nombres de las credenciales que están vacías o no existen
# 'if not v' detecta tanto None (variable inexistente) como '' (variable vacía)
vacias = [k for k, v in {
    "ACCOUNT": ACCOUNT, "USER": USER, "PASSWORD": PASSWORD,
    "WAREHOUSE": WAREHOUSE, "DATABASE": DATABASE, "SCHEMA": SCHEMA
}.items() if not v]

# Si falta alguna credencial, lanzamos un error descriptivo antes de intentar conectar
if vacias:
    raise ValueError(f"Faltan credenciales en .env: {vacias}")

print(f"Cuenta:    {ACCOUNT}")
print(f"Usuario:   {USER}")
print(f"Warehouse: {WAREHOUSE}")
print(f"Base:      {DATABASE}.{SCHEMA}")
print("Credenciales cargadas correctamente.")

Cuenta:    aisfeil-lu45385
Usuario:   socratesagudo
Warehouse: FRAUDE_VILAGARCIA
Base:      FRAUDE_VILAGARCIA.PUBLIC
Credenciales cargadas correctamente.


## 2 — Conectar a Snowflake

In [17]:
# Abrimos la conexión con Snowflake usando las credenciales del .env
# snowflake.connector.connect() devuelve un objeto de conexión activa
conn = snowflake.connector.connect(
    account   = ACCOUNT,
    user      = USER,
    password  = PASSWORD,
    warehouse = WAREHOUSE,
    database  = DATABASE,
    schema    = SCHEMA,
)

# El cursor es el objeto intermediario que envía SQL a Snowflake y recibe resultados
# La conexión abre el canal; el cursor es el 'lápiz' con el que escribes dentro de ese canal
cur = conn.cursor()

print("Conexión establecida.")
# Ejecutamos una query inofensiva como test de conectividad
cur.execute("SELECT CURRENT_VERSION()")
# fetchone() recupera la primera (y única) fila del resultado como tupla
# [0] extrae el primer elemento de esa tupla — el string con la versión
version = cur.fetchone()[0]
print(f"Versión de Snowflake: {version}")

Conexión establecida.
Versión de Snowflake: 10.14.103


## 3 — Crear el esquema en estrella

Se crean 2 tablas de dimensiones y 1 tabla de hechos. Se descartan columnas sin valor analítico:
- `codigo_cpv` — 100% nulo
- `tipo_procedimiento`, `sistema_contratacion`, `tramitacion` — siempre constantes (son resultado del filtro de Contratos Menores)

| Tabla | Tipo | Descripción |
|---|---|---|
| `DIM_CONTRATISTA` | Dimensión | Empresas y autónomos que reciben contratos |
| `DIM_FECHA` | Dimensión | Calendario con atributos temporales |
| `FACT_CONTRATOS` | Hechos | Métricas económicas por contrato |

In [18]:
# Limpieza previa: borramos todas las tablas del proyecto antes de recrearlas
# Esto garantiza que cada ejecución parte de un estado limpio y reproducible
# DROP TABLE IF EXISTS no lanza error si la tabla no existe — es seguro ejecutarlo siempre
# Incluimos también tablas de versiones anteriores del notebook para no dejar huérfanas en Snowflake
TABLAS_PROYECTO = [
    "FACT_CONTRATOS",
    "DIM_CONTRATISTA",
    "DIM_FECHA",
    "DIM_ENTIDAD",         # eliminada — se conservaba en versiones anteriores
    "DIM_TIPO_CONTRATO",   # eliminada — se conservaba en versiones anteriores
]

for tabla in TABLAS_PROYECTO:
    cur.execute(f"DROP TABLE IF EXISTS {tabla}")
    print(f"  {tabla:<22} eliminada (si existía).")

  FACT_CONTRATOS         eliminada (si existía).
  DIM_CONTRATISTA        eliminada (si existía).
  DIM_FECHA              eliminada (si existía).
  DIM_ENTIDAD            eliminada (si existía).
  DIM_TIPO_CONTRATO      eliminada (si existía).


In [19]:
# Definimos el esquema de las 2 dimensiones y la tabla de hechos
# DDL = Data Definition Language: el subconjunto de SQL que define estructura (no datos)
# Usamos CREATE TABLE (sin OR REPLACE) porque la limpieza previa ya borró las tablas

# DIM_CONTRATISTA: una fila por empresa o autónomo que ha recibido al menos un contrato
DDL_DIM_CONTRATISTA = """
CREATE TABLE DIM_CONTRATISTA (
    nif_contratista    VARCHAR PRIMARY KEY,  -- NIF o CIF del contratista
    nombre_contratista VARCHAR,              -- razón social o nombre
    nif_valido         BOOLEAN,              -- True si pasa la validación de checksum
    motivo_invalido    VARCHAR,              -- descripción del error si nif_valido = False
    contratistas_raw   VARCHAR               -- campo original para trazabilidad
)
"""

# DIM_FECHA: calendario continuo con una fila por día entre la fecha mínima y máxima del dataset
# Power BI exige esta continuidad para poder marcarla como tabla de fechas e inteligencia temporal
DDL_DIM_FECHA = """
CREATE TABLE DIM_FECHA (
    fecha       DATE PRIMARY KEY,  -- clave natural — la fecha completa
    anio        INTEGER,
    trimestre   INTEGER,           -- 1 a 4
    mes         INTEGER,           -- 1 a 12
    nombre_mes  VARCHAR,           -- 'Enero', 'Febrero'...
    dia_mes     INTEGER,           -- 1 a 31
    dia_semana  VARCHAR            -- 'Lunes', 'Martes'...
)
"""

# FACT_CONTRATOS: una fila por contrato — contiene las métricas económicas y las claves foráneas
# Las columnas marcadas FK apuntan a registros de las dimensiones
# La FK temporal se llama 'fecha' (mismo nombre que la PK de DIM_FECHA) para que Power BI
# detecte la relación automáticamente al importar el modelo
DDL_FACT = """
CREATE TABLE FACT_CONTRATOS (
    num_referencia       VARCHAR PRIMARY KEY,  -- identificador único del expediente
    anio_contrato        INTEGER,
    num_contrato_anio    INTEGER,              -- secuencial dentro del año
    fecha_formalizacion  DATE,                 -- fecha real (99.5% null)
    fecha                DATE,                 -- FK → DIM_FECHA (sintética — siempre rellena)
    tipo_contrato        VARCHAR,              -- Obras / Servicios / Suministros / Privado
    nif_contratista      VARCHAR,              -- FK → DIM_CONTRATISTA
    objeto_contrato      VARCHAR,
    plazo_meses          FLOAT,
    valor_estimado_eur   FLOAT,
    presupuesto_base_eur FLOAT,
    importe_sin_iva_eur  FLOAT,
    importe_con_iva_eur  FLOAT,
    flag_limite          VARCHAR,              -- ok / cerca_del_limite / supera_limite
    observaciones        VARCHAR,
    url_licitacion       VARCHAR
)
"""

# Ejecutamos cada DDL en Snowflake en el orden correcto (primero dimensiones, luego fact)
for nombre, ddl in [
    ("DIM_CONTRATISTA", DDL_DIM_CONTRATISTA),
    ("DIM_FECHA",       DDL_DIM_FECHA),
    ("FACT_CONTRATOS",  DDL_FACT),
]:
    cur.execute(ddl)
    print(f"Tabla {nombre} creada correctamente.")

Tabla DIM_CONTRATISTA creada correctamente.
Tabla DIM_FECHA creada correctamente.
Tabla FACT_CONTRATOS creada correctamente.


## 4 — Cargar el CSV y construir las dimensiones

In [ ]:
# Leemos el CSV limpio producido por el notebook 02 — ya incluye fecha_estimada generada con semilla fija
RUTA_CSV = Path("../data/processed/contratos_limpios.csv")

df = pd.read_csv(
    RUTA_CSV,
    encoding    = "utf-8-sig",   # respeta el BOM que añadió el notebook de transformación
    parse_dates = ["fecha_formalizacion", "fecha_estimada"],  # ambas columnas a Timestamp
    dtype       = {"anio_contrato": "Int64", "num_contrato_anio": "Int64"},  # Int64 con mayúscula admite nulos
)

print(f"Filas cargadas: {len(df)}")
df.head(3)

In [21]:
# ── DIM_CONTRATISTA ──────────────────────────────────────────────────────
# Seleccionamos solo las columnas que pertenecen a esta dimensión
dim_contratista = (
    df[["nif_contratista", "nombre_contratista", "nif_valido", "motivo_invalido", "contratistas_raw"]]
    .dropna(subset=["nif_contratista"])          # descartamos filas sin NIF — no pueden ser contratistas válidos
    .drop_duplicates(subset=["nif_contratista"])  # un mismo NIF puede aparecer en varios contratos; nos quedamos con la primera aparición
    .reset_index(drop=True)                       # reindexamos desde 0 después de eliminar filas
)

# ── DIM_FECHA ─────────────────────────────────────────────────────────────
# Generamos un CALENDARIO COMPLETO Y CONTINUO entre la fecha mínima y máxima del dataset
# Power BI exige continuidad (todos los días sin huecos) para poder marcar la tabla como tabla de fechas
# pd.date_range(inicio, fin, freq='D') produce un DatetimeIndex con un Timestamp por día
fecha_min = df["fecha_estimada"].min()
fecha_max = df["fecha_estimada"].max()
rango_fechas = pd.date_range(start=fecha_min, end=fecha_max, freq="D")

print(f"Rango de fechas: {fecha_min.date()} → {fecha_max.date()}  ({len(rango_fechas)} días)")

# Convertimos el DatetimeIndex a Serie para poder usar .dt sobre él de forma idéntica a antes
todas_fechas = pd.Series(rango_fechas)

# Diccionarios de traducción: número → nombre en español
# Python numera los meses desde 1 y los días desde 0 (0 = Lunes)
MESES_ES = {
    1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril",
    5: "Mayo", 6: "Junio", 7: "Julio", 8: "Agosto",
    9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre",
}
DIAS_ES = {
    0: "Lunes", 1: "Martes", 2: "Miercoles", 3: "Jueves",
    4: "Viernes", 5: "Sabado", 6: "Domingo",
}

# Construimos el DataFrame de DIM_FECHA descomponiendo cada Timestamp en sus atributos
# .dt. permite acceder a propiedades de fecha sobre una serie entera a la vez
dim_fecha = pd.DataFrame({
    "fecha":      todas_fechas.dt.date,                      # convierte Timestamp a date nativo de Python (sin hora)
    "anio":       todas_fechas.dt.year,                      # extrae el año (ej. 2023)
    "trimestre":  todas_fechas.dt.quarter,                   # 1 = ene-mar, 2 = abr-jun, etc.
    "mes":        todas_fechas.dt.month,                     # número de mes (1-12)
    "nombre_mes": todas_fechas.dt.month.map(MESES_ES),       # sustituye el número por el nombre en español
    "dia_mes":    todas_fechas.dt.day,                       # día del mes (1-31)
    "dia_semana": todas_fechas.dt.dayofweek.map(DIAS_ES),    # sustituye el número de día (0-6) por su nombre
})

print(f"DIM_CONTRATISTA: {len(dim_contratista)} filas")
print(f"DIM_FECHA:       {len(dim_fecha)} filas (calendario continuo)")

Rango de fechas: 2021-02-04 → 2023-12-31  (1061 días)
DIM_CONTRATISTA: 334 filas
DIM_FECHA:       1061 filas (calendario continuo)


## 5 — Insertar dimensiones en Snowflake

In [22]:
import numpy as np

def normalizar(valor):
    """Convierte tipos pandas/numpy a tipos Python nativos que acepta el conector de Snowflake."""
    # El conector de Snowflake solo acepta tipos nativos de Python (int, float, bool, date...)
    # pandas y numpy usan sus propios tipos internos que el conector no reconoce
    if valor is None:
        return None
    try:
        if pd.isna(valor):   # detecta NaN, NaT y pd.NA — todos se convierten a NULL en Snowflake
            return None
    except (TypeError, ValueError):
        pass  # algunos tipos no son comparables con isna(); los dejamos pasar al siguiente check
    if isinstance(valor, pd.Timestamp):
        return valor.date()       # Timestamp (con hora) → date (solo fecha)
    if isinstance(valor, np.integer):
        return int(valor)         # numpy int64/int32/etc. → int nativo de Python
    if isinstance(valor, np.floating):
        return float(valor)       # numpy float64/etc. → float nativo de Python
    if isinstance(valor, np.bool_):
        return bool(valor)        # numpy bool_ → bool nativo de Python
    return valor                  # strings y otros tipos ya son nativos de Python

def insertar_tabla(nombre_tabla, df_dim, batch_size=1000):
    """Inserta un DataFrame completo en Snowflake en lotes para no superar el límite de mensaje."""
    # Construimos el SQL de inserción dinámicamente a partir de los nombres de columna del DataFrame
    columnas  = ", ".join(df_dim.columns)           # 'col1, col2, col3'
    wildcards = ", ".join(["%s"] * len(df_dim.columns))  # '%s, %s, %s' — placeholders para los valores
    sql = f"INSERT INTO {nombre_tabla} ({columnas}) VALUES ({wildcards})"

    # Convertimos el DataFrame a una lista de tuplas con todos los valores normalizados
    # itertuples() es más rápido que iterrows() para iterar fila a fila
    filas = [
        tuple(normalizar(v) for v in row)
        for row in df_dim.itertuples(index=False, name=None)
    ]

    # Insertamos en lotes de batch_size filas para no superar el límite de tamaño de mensaje
    # executemany() es más eficiente que ejecutar un INSERT por fila
    for i in range(0, len(filas), batch_size):
        cur.executemany(sql, filas[i : i + batch_size])

    conn.commit()  # confirmamos la transacción — sin commit los datos no se persisten
    print(f"  {nombre_tabla:<22} {len(filas)} filas insertadas.")

# Las dimensiones se cargan primero para que las claves foráneas de FACT_CONTRATOS ya existan
print("Insertando dimensiones:")
insertar_tabla("DIM_CONTRATISTA", dim_contratista)
insertar_tabla("DIM_FECHA",       dim_fecha)
print("Dimensiones cargadas correctamente.")


Insertando dimensiones:
  DIM_CONTRATISTA        334 filas insertadas.
  DIM_FECHA              1061 filas insertadas.
Dimensiones cargadas correctamente.


## 6 — Construir e insertar la tabla de hechos

In [23]:
# Hacemos una copia para no modificar el DataFrame original (df) que aún necesitamos para la verificación
df_fact = df.copy()

# Renombramos la columna fecha_estimada → fecha
# Esto alinea el nombre de la FK en FACT_CONTRATOS con la PK de DIM_FECHA ('fecha'),
# lo que permite que Power BI detecte la relación automáticamente al importar el modelo
df_fact = df_fact.rename(columns={"fecha_estimada": "fecha"})

# Convertimos ambas columnas de fecha a datetime.date nativo de Python
# El conector de Snowflake no acepta pd.Timestamp — solo acepta datetime.date
for col in ["fecha_formalizacion", "fecha"]:
    df_fact[col] = df_fact[col].apply(lambda x: x.date() if pd.notna(x) else None)

# Convertimos enteros nullable de pandas (Int64) a int nativo o None
# select_dtypes detecta todas las columnas de tipo entero nullable automáticamente
for col in df_fact.select_dtypes(include=["Int64", "Int32"]).columns:
    df_fact[col] = df_fact[col].apply(lambda x: None if pd.isna(x) else int(x))

# Reemplazamos cualquier NA/NaN restante por None
# df.where() mantiene los valores donde la condición es True y sustituye el resto por 'other'
df_fact = df_fact.where(pd.notna(df_fact), other=None)

# Seleccionamos solo las columnas de FACT_CONTRATOS en el mismo orden que el DDL
# Esto descarta columnas que van en las dimensiones (nombre_contratista, nif_valido, etc.)
COLS_FACT = [
    "num_referencia", "anio_contrato", "num_contrato_anio",
    "fecha_formalizacion", "fecha",
    "tipo_contrato", "nif_contratista",
    "objeto_contrato", "plazo_meses",
    "valor_estimado_eur", "presupuesto_base_eur", "importe_sin_iva_eur", "importe_con_iva_eur",
    "flag_limite", "observaciones", "url_licitacion",
]
df_fact = df_fact[COLS_FACT]

print(f"Insertando {len(df_fact)} filas en FACT_CONTRATOS...")
insertar_tabla("FACT_CONTRATOS", df_fact)

Insertando 607 filas en FACT_CONTRATOS...
  FACT_CONTRATOS         607 filas insertadas.


## 7 — Verificación de la carga

In [24]:
# Comprobamos que el número de filas en FACT_CONTRATOS coincide exactamente con el CSV
# Si no coinciden, hubo algún error en la carga que hay que investigar
cur.execute("SELECT COUNT(*) FROM FACT_CONTRATOS")
n_fact = cur.fetchone()[0]  # fetchone() devuelve la primera fila como tupla; [0] extrae el número
print(f"Filas en CSV:            {len(df)}")
print(f"Filas en FACT_CONTRATOS: {n_fact}")
print(f"Coinciden: {'OK' if len(df) == n_fact else 'ERROR — revisar'}")
print()

# Mostramos el conteo de cada dimensión para verificar que se cargaron correctamente
for tabla in ["DIM_CONTRATISTA", "DIM_FECHA"]:
    cur.execute(f"SELECT COUNT(*) FROM {tabla}")
    n = cur.fetchone()[0]
    print(f"  {tabla:<22} {n} filas")

Filas en CSV:            607
Filas en FACT_CONTRATOS: 607
Coinciden: OK

  DIM_CONTRATISTA        334 filas
  DIM_FECHA              1061 filas


In [25]:
# Vista previa con JOIN entre la fact y las dimensiones
# Simulamos exactamente cómo Power BI consultará los datos al construir un informe
# Usamos INNER JOIN porque la columna 'fecha' está rellena en todas las filas (generada en notebook 02)
cur.execute("""
    SELECT
        f.num_referencia,
        d.anio,
        d.nombre_mes,
        f.tipo_contrato,
        f.presupuesto_base_eur,
        c.nombre_contratista,
        c.nif_valido,
        f.flag_limite
    FROM FACT_CONTRATOS f
    JOIN DIM_FECHA       d ON f.fecha           = d.fecha           -- une el contrato con los atributos de su fecha
    JOIN DIM_CONTRATISTA c ON f.nif_contratista = c.nif_contratista -- une el contrato con los datos del contratista
    LIMIT 5
""")
# fetchall() recupera todas las filas del resultado como lista de tuplas
muestra = cur.fetchall()

cabecera = ["num_referencia", "anio", "mes", "tipo", "presupuesto", "contratista", "nif_ok", "flag"]
print("Muestra con JOIN entre tablas:")
print("  " + "  |  ".join(f"{c:<15}" for c in cabecera))
print("  " + "-" * 105)
for fila in muestra:
    print("  " + "  |  ".join(f"{str(v):<15}" for v in fila))

Muestra con JOIN entre tablas:
  num_referencia   |  anio             |  mes              |  tipo             |  presupuesto      |  contratista      |  nif_ok           |  flag           
  ---------------------------------------------------------------------------------------------------------
  CT 20/2021-e     |  2021             |  Febrero          |  Obras            |  35351.6          |  MOVIMIENTO DE ARIDOS Y CONSTRUCCIONES DE AROSA S. L.  |  True             |  ok             
  CT 97/2021-e     |  2021             |  Agosto           |  Obras            |  23093.95         |  Montajes J.M. Iglesias, S.L.  |  True             |  ok             
  CT 80/2021-e     |  2021             |  Octubre          |  Obras            |  28613.18         |  Montajes J.M. Iglesias, S.L.  |  True             |  ok             
  PRP2022/4631     |  2022             |  Enero            |  Servicios        |  3000.0           |  IMATIA INNOVATION S.L.  |  True             |  ok             
 

## 8 — Cerrar la conexión

In [26]:
# Cerramos el cursor y la conexión para liberar los recursos en Snowflake
# Si no se cierran, la sesión permanece abierta hasta que expire por timeout
cur.close()
conn.close()
print("Conexión cerrada.")

Conexión cerrada.
